# Lección 7.1 — Regularización: castigar la complejidad

> **Módulo 7 · Regularización y ajuste de hiperparámetros** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 5.1 (regresión lineal), Lección 5.2 (regresión logística), Lección 4.4 (sobreajuste vs. subajuste), Lección 4.5 (sesgo-varianza)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía qué es **regularizar** y por qué reduce el sobreajuste.
- Distinguir **L2 (Ridge)**, **L1 (Lasso)** y **Elastic Net** por lo que le hacen a los coeficientes.
- Predecir qué pasa al subir o bajar `alpha` (y su "primo al revés", `C`).
- Evitar los dos errores clásicos: regularizar **sin escalar** y confundir `alpha` con `C`.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression, RidgeCV, LassoCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.datasets import make_regression, load_diabetes, load_breast_cancer

warnings.filterwarnings("ignore")          # ocultamos avisos de convergencia para no distraernos
plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: la aerolínea que cobra por kilo

Imagina que viajas y la aerolínea **cobra por cada kilo de equipaje**. ¿Qué haces?
- Sigues llevando lo **imprescindible** (pasaporte, cargador).
- Dejas en casa lo que "por si acaso" metiste (el tercer par de zapatos, la plancha de viaje).

Sin ese cobro, la maleta crece sin control. Con el cobro, solo viaja lo que **vale lo que pesa**.

Un modelo es igual. Sus "kilos" son los **coeficientes** (las perillas de la Lección 3.1). Sin restricciones, un modelo flexible usa coeficientes enormes para acomodar **cada punto** de entrenamiento, incluido el ruido: **sobreajuste** (Lección 4.4).

**Regularizar = cobrarle al modelo por la complejidad.** Su nueva meta es:

```
   pérdida total  =  error en los datos   +   alpha × "peso" de los coeficientes
                     (lo de siempre)          (el cobro por kilo)
```

`alpha` es la **tarifa**: con `alpha = 0` viajas gratis (modelo sin regularizar); con un `alpha` enorme no te llevas nada (el modelo se queda casi plano).

### ✋ Ejercicio 1 — Calentamiento
Con la tarifa por kilo, ¿qué objeto es más probable que **dejes** en casa?
- a) Un objeto pesado que usarás todos los días
- b) Un objeto pesado que "quizá" uses una vez
- c) Un objeto ligerísimo que usarás todos los días

¿Y qué tiene que ver eso con un coeficiente que solo sirve para acomodar **un** punto raro del entrenamiento?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**b)** Pesa mucho y aporta poco. El cobro te obliga a comparar **costo** (peso) contra **beneficio** (uso).

Un coeficiente grande que solo sirve para acomodar un punto raro es exactamente eso: **mucho peso, poco beneficio**. La regularización hace que al modelo "no le convenga" llevarlo.
</details>

---
## 2. 📖 El concepto: el mismo modelo, con y sin cobro

Vamos a verlo. Tenemos solo **15 puntos** que siguen una curva suave con ruido, y un modelo **muy flexible**: un polinomio de grado 12 (una regresión lineal con 12 "versiones" de `x`: `x`, `x²`, `x³`…).

- **Sin regularizar** (`LinearRegression`): nadie le cobra por los coeficientes.
- **Con regularización L2** (`Ridge`, `alpha = 0.01`): mismo modelo, pero con tarifa.

> 📝 El paso `StandardScaler` pone todas las columnas en la misma escala. En la Sección 6 verás por qué es **obligatorio** al regularizar.

In [ ]:
rng = np.random.default_rng(0)
x_tr = np.sort(rng.uniform(0, 1, 15)).reshape(-1, 1)        # 15 puntos de entrenamiento
x_te = np.sort(rng.uniform(0, 1, 200)).reshape(-1, 1)       # 200 puntos "nuevos" para probar
curva_real = lambda x: np.sin(2 * np.pi * x).ravel()
y_tr = curva_real(x_tr) + rng.normal(0, 0.3, 15)
y_te = curva_real(x_te) + rng.normal(0, 0.3, 200)

def polinomio(modelo, grado=12):
    return make_pipeline(PolynomialFeatures(grado, include_bias=False), StandardScaler(), modelo)

modelos = {"Sin regularizar": polinomio(LinearRegression()),
           "Ridge (alpha=0.01)": polinomio(Ridge(alpha=0.01))}

xx = np.linspace(0, 1, 300).reshape(-1, 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (nombre, m) in zip(axes, modelos.items()):
    m.fit(x_tr, y_tr)
    coefs = m[-1].coef_
    ax.scatter(x_tr, y_tr, c="k", zorder=3, label="entrenamiento")
    ax.plot(xx, curva_real(xx), "g--", alpha=.6, label="curva real")
    ax.plot(xx, m.predict(xx), "r", label="modelo")
    ax.set_ylim(-2.5, 2.5); ax.legend(fontsize=8)
    ax.set_title(f"{nombre}\nerror train {mean_squared_error(y_tr, m.predict(x_tr)):.3f} · "
                 f"error test {mean_squared_error(y_te, m.predict(x_te)):.3f}\n"
                 f"coeficiente más grande: {np.abs(coefs).max():,.0f}", fontsize=10)
plt.tight_layout(); plt.show()

### ✋ Ejercicio 2 — Lee las gráficas
1. ¿Cuál de los dos tiene **menor error en entrenamiento**? ¿Y en prueba?
2. Mira el **coeficiente más grande** de cada uno. ¿Qué relación ves entre coeficientes enormes y una curva "loca"?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. El modelo **sin regularizar** tiene el error de entrenamiento más bajo (≈ 0.08: pasa casi por todos los puntos), pero su error de prueba es **muchísimo** peor (≈ 7.6, ¡casi 70 veces el de Ridge!). Ridge tiene un error de entrenamiento un poco más alto (≈ 0.12) y un error de prueba mucho más bajo (≈ 0.11): **renunció a memorizar para generalizar**.
2. El modelo sin regularizar necesita coeficientes **de miles** (el mayor ronda los 2,400) que se cancelan entre sí para zigzaguear entre los puntos. Ridge se mantiene con coeficientes **pequeños** (el mayor ronda 2), y eso produce una curva suave. **Coeficientes enormes = curva nerviosa = sobreajuste.**
</details>

---
## 3. 📖 Cómo funciona: dos formas de cobrar (L2 y L1)

La diferencia entre los métodos está en **cómo** se mide el "peso" de los coeficientes:

| Método | Cómo cobra | Analogía | Efecto típico |
|---|---|---|---|
| **L2 · Ridge** | Suma de coeficientes **al cuadrado** | Cobro que se dispara con los objetos muy pesados | **Encoge** todos los coeficientes, casi nunca los deja en cero |
| **L1 · Lasso** | Suma de coeficientes **en valor absoluto** | Cobro fijo por kilo, también por los kilos pequeños | Lleva muchos coeficientes **exactamente a cero**: elimina variables |
| **Elastic Net** | Una mezcla de los dos | Un poco de cada | Elimina variables, pero con más estabilidad que Lasso |

¿Por qué L1 deja ceros y L2 no? Intuición:
- Con **L2** (cuadrado), un coeficiente de 0.1 cuesta 0.01: casi nada. Al modelo no le "urge" bajarlo a cero; solo encoge los grandes.
- Con **L1** (valor absoluto), un coeficiente de 0.1 sigue costando 0.1. Si esa variable aporta poco, **sale más barato apagarla del todo**.

Por eso Lasso hace **selección de variables** automática. Comprobémoslo con datos donde sabemos la verdad: **20 variables, de las cuales solo 5 sirven** (las otras 15 son ruido).

### 🔮 Ejercicio 3 — ¿Cuántos ceros?
Vamos a entrenar tres modelos sobre esos datos y contar cuántos coeficientes quedan **exactamente en cero**.

Antes de ejecutar, predice para cada uno: ¿cerca de 0, cerca de 5 o cerca de 15 ceros?
- `LinearRegression` → ___
- `Ridge(alpha=10)` → ___
- `Lasso(alpha=5)` → ___

✍️ *Mis predicciones:*

In [ ]:
X20, y20, coef_real = make_regression(n_samples=100, n_features=20, n_informative=5,
                                      noise=10, coef=True, random_state=0)
utiles = np.flatnonzero(coef_real)          # las 5 variables que de verdad influyen
print("Variables útiles de verdad:", utiles.tolist())

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
for ax, m in zip(axes, [LinearRegression(), Ridge(alpha=10), Lasso(alpha=5)]):
    m.fit(X20, y20)
    colores = ["tab:green" if i in utiles else "tab:gray" for i in range(20)]
    ax.bar(range(20), m.coef_, color=colores)
    ceros = int(np.sum(m.coef_ == 0))
    ax.set_title(f"{type(m).__name__}: {ceros} coeficientes en cero", fontsize=10)
    ax.set_xlabel("variable (verde = útil de verdad)")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- `LinearRegression` → **0** ceros: le da algún peso, aunque sea pequeño, a todas.
- `Ridge(alpha=10)` → **0** ceros: encoge, pero no apaga nada. Las barras grises son pequeñas (como mucho ≈ 4), pero existen.
- `Lasso(alpha=5)` → **15** ceros: se queda **exactamente** con las 5 útiles y apaga todas las grises. **Descubrió sola cuáles variables importan.**

Ojo: Lasso no es infalible. Con más ruido o menos datos puede quedarse con alguna variable falsa o tirar una útil. Pero es una ayuda enorme cuando sospechas que muchas columnas sobran.
</details>

---
## 4. 📖 ¿Qué pasa si subo la tarifa? El camino de los coeficientes

Ahora usamos datos reales: el progreso de la **diabetes** en 442 pacientes, con 10 variables (edad, índice de masa corporal, presión, varias mediciones en sangre…).

Vamos a subir `alpha` poco a poco, desde casi 0 hasta un valor enorme, y dibujar **cómo cambia cada coeficiente**. Cada línea es una variable.

### 🔮 Ejercicio 4 — Antes de ver la gráfica
1. Cuando `alpha` se hace **enorme**, ¿a qué valor tienden **todos** los coeficientes, en Ridge y en Lasso?
2. ¿Qué diferencia esperas en **la forma** en que llegan ahí?

✍️ *Mis predicciones:*

In [ ]:
X_diab, y_diab = load_diabetes(return_X_y=True, as_frame=True)
X_diab_esc = StandardScaler().fit_transform(X_diab)

alphas = np.logspace(-2, 4, 60)          # de 0.01 a 10,000, en escala logarítmica
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, Modelo, alphas_m in [(axes[0], Ridge, alphas), (axes[1], Lasso, alphas / 100)]:
    caminos = np.array([Modelo(alpha=a).fit(X_diab_esc, y_diab).coef_ for a in alphas_m])
    for j, col in enumerate(X_diab.columns):
        ax.plot(alphas_m, caminos[:, j], label=col)
    ax.axhline(0, color="k", lw=.8); ax.set_xscale("log")
    ax.set_xlabel("alpha (escala logarítmica) →  más castigo"); ax.set_ylabel("coeficiente")
    ax.set_title(f"{Modelo.__name__}: camino de los coeficientes")
axes[1].legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. A **cero**. Con una tarifa altísima, cualquier coeficiente cuesta más de lo que aporta, y el modelo termina prediciendo siempre el promedio: **subajuste** total.
2. **Ridge** baja todas las líneas **suavemente** y a la vez; tocan el cero solo al final, casi asintóticamente. **Lasso** las va **apagando una por una**: cada línea llega a cero en un punto concreto y se queda ahí. Las últimas en apagarse (`bmi`, `s5`) son las que el modelo considera más importantes.

> 📝 Los valores de `alpha` de Lasso y Ridge no son comparables entre sí (miden el "peso" de forma distinta), por eso cada gráfica usa su propio rango.
</details>

### 📖 Ni poco ni mucho: la "U" de siempre
`alpha` es una perilla de **sesgo-varianza** (Lección 4.5):
- `alpha` muy bajo → modelo libre → **varianza alta** (sobreajuste).
- `alpha` muy alto → modelo amordazado → **sesgo alto** (subajuste).

El mejor valor está en medio, y se elige con **validación cruzada** (Lección 4.6), nunca mirando el conjunto de prueba.

### ✏️ Ejercicio 5 — Encuentra la tarifa justa
Volvemos al polinomio de grado 12 del principio, pero ahora con **30 puntos**. La celda calcula el error de validación cruzada para cada `alpha`. Ejecuta y elige el mejor. Luego ✏️ cambia `GRADO` a 3 y vuelve a ejecutar.

In [ ]:
GRADO = 12      # ✏️ prueba también 3

x30 = rng.uniform(0, 1, 30).reshape(-1, 1)
y30 = curva_real(x30) + rng.normal(0, 0.3, 30)

for a in [1e-6, 1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]:
    error = -cross_val_score(polinomio(Ridge(alpha=a), GRADO), x30, y30, cv=5,
                             scoring="neg_mean_squared_error").mean()
    print(f"alpha={a:<7g} error CV = {error:6.3f}  " + "█" * int(min(error, 1) * 60))

1. ¿Qué `alpha` da el menor error con `GRADO = 12`? ¿Qué pasa en los extremos?
2. Con `GRADO = 3`, ¿la regularización ayuda tanto? ¿Por qué?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. El mínimo está en **`alpha` = 0.01** (error ≈ 0.19). Con `alpha` casi nulo (1e-6) el error es enorme (≈ 0.68): sobreajuste. Con `alpha = 100` vuelve a subir (≈ 0.41): subajuste. Es la **"U"**.
2. Con grado 3, sin regularizar el error ya es mucho menor (≈ 0.32 contra 0.68): un modelo sencillo sobreajusta menos, así que la regularización tiene **menos que arreglar** (su mejor valor, ≈ 0.22, también está en `alpha = 0.01`). Curiosidad: el polinomio de grado 12 **bien regularizado** (0.19) le gana al de grado 3. **Un modelo flexible con la tarifa justa suele ser una gran combinación.**
</details>

---
## 5. 📖 Elastic Net: lo mejor de los dos mundos

Lasso tiene una manía: cuando dos variables están **muy correlacionadas** (dicen casi lo mismo, como "metros cuadrados" y "número de habitaciones"), suele quedarse con **una** y tirar la otra, casi al azar. Si cambian un poco los datos, puede elegir la otra. Eso es incómodo si quieres interpretar el modelo.

**Elastic Net** mezcla L1 y L2 con la perilla `l1_ratio`:
- `l1_ratio = 1` → Lasso puro.
- `l1_ratio = 0` → Ridge puro.
- En medio → elimina variables inútiles (L1) pero **reparte** el peso entre las correlacionadas (L2).

### 🔮 Ejercicio 6 — Las gemelas
Creamos dos variables casi idénticas, `gemela_A` y `gemela_B`, que juntas influyen en `y`, más una variable de ruido. ¿Qué crees que hará cada modelo con las gemelas?

✍️ *Mi predicción:*

In [ ]:
rng_g = np.random.default_rng(1)
n = 200
base = rng_g.normal(size=n)
gemelas = pd.DataFrame({
    "gemela_A": base + rng_g.normal(0, 0.05, n),
    "gemela_B": base + rng_g.normal(0, 0.05, n),     # casi la misma información
    "ruido":    rng_g.normal(size=n),
})
y_g = 3 * base + rng_g.normal(0, 1, n)

tabla = pd.DataFrame({
    "Lasso":                    Lasso(alpha=0.1).fit(gemelas, y_g).coef_,
    "ElasticNet (l1_ratio=0.5)": ElasticNet(alpha=0.1, l1_ratio=0.5).fit(gemelas, y_g).coef_,
    "Ridge":                    Ridge(alpha=10).fit(gemelas, y_g).coef_,
}, index=gemelas.columns).round(2)
tabla

<details><summary><b>👀 Ver respuesta</b></summary>

- **Lasso** carga **todo** el peso en una gemela (`gemela_B` ≈ 2.92) y deja la otra en **cero**. Eligió una favorita.
- **Elastic Net** reparte el peso entre las dos (≈ 1.37 y 1.53).
- **Ridge** también reparte (≈ 1.40 y 1.55).
- Todas mandan el **ruido** a cero o casi cero.

Si para tu negocio importa decir "estas dos variables importan", Elastic Net es más honesto que Lasso. Si solo quieres el modelo más pequeño posible, Lasso está bien.
</details>

---
## 6. 📖 Hiperparámetros y perillas

| Hiperparámetro | Modelo | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|---|
| `alpha` | `Ridge`, `Lasso`, `ElasticNet` | La tarifa por complejidad | Coeficientes más pequeños → riesgo de **subajuste** | Modelo más libre → riesgo de **sobreajuste** | 1.0 y buscar en escala log (0.001 – 1000) |
| `l1_ratio` | `ElasticNet` | Mezcla L1 / L2 | Más Lasso: más ceros | Más Ridge: reparte el peso | 0.5 |
| `C` | `LogisticRegression`, `SVC` | **El inverso** de la tarifa (`C ≈ 1/alpha`) | **Menos** castigo → riesgo de sobreajuste | **Más** castigo → riesgo de subajuste | 1.0 y buscar en escala log (0.001 – 1000) |

### 🧠 Regla mnemotécnica
> **`alpha` es la multa; `C` es la libertad.** Subir la multa simplifica. Subir la libertad complica.

### 🔮 Ejercicio 7 — `C` va al revés
En el diagnóstico de cáncer de mama (Lección 5.4), entrenamos regresiones logísticas con distintos valores de `C`. La columna `peso_total` suma el tamaño de todos los coeficientes.

Antes de ejecutar: ¿con qué `C` esperas el `peso_total` **más pequeño**? ¿Y la mayor brecha entre entrenamiento y prueba?

✍️ *Mi predicción:*

In [ ]:
Xc, yc = load_breast_cancer(return_X_y=True)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, random_state=42)

for C in [0.001, 0.01, 0.1, 1, 10, 1000]:      # ✏️ prueba otros valores
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(Xc_tr, yc_tr)
    print(f"C={C:<6}  peso_total={np.abs(m[-1].coef_).sum():7.1f}   "
          f"train={m.score(Xc_tr, yc_tr):.1%}   test={m.score(Xc_te, yc_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

- El `peso_total` más pequeño es con **`C = 0.001`** (≈ 1.4): mucho castigo, coeficientes diminutos. Con `C = 1000` llega a ≈ 294.
- La mayor brecha aparece con `C = 1000`: **100 % en entrenamiento** y ≈ 94.4 % en prueba. Con `C = 0.001`, el modelo está tan amordazado que baja en ambos (≈ 89–90 %): subajuste.
- El punto dulce aquí está en `C` ≈ 0.1 (≈ 98.6 % en prueba).

`C` grande = **poca** regularización. Es justo al revés que `alpha`. Recuérdalo, porque es la confusión número uno.

> 📝 Por defecto, `LogisticRegression` usa L2. Para usar L1 (con ceros, como Lasso): en scikit-learn ≥ 1.8 escribe `LogisticRegression(l1_ratio=1, solver="saga")`; en versiones anteriores, `LogisticRegression(penalty="l1", solver="saga")`.
</details>

### 📖 La regularización está en todas partes
Ridge y Lasso le ponen nombre, pero **casi todos los modelos** tienen alguna perilla que "cobra por complejidad":

| Modelo | Perillas que regularizan | Lección |
|---|---|---|
| Regresión lineal / logística | `alpha`, `C`, `l1_ratio` | 5.1, 5.2, esta |
| SVM | `C` (y `gamma` en kernels) | 5.5 |
| Árbol de decisión | `max_depth`, `min_samples_leaf`, `ccp_alpha` | 5.4 |
| Gradient boosting | `learning_rate`, `max_depth`, `l2_regularization`, `subsample` | 6.3, 6.4 |
| Redes neuronales | `alpha` (weight decay), dropout, **early stopping** | 7.2, 9.8 |

La idea es siempre la misma: **impedir que el modelo use toda su flexibilidad para memorizar.**

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Lasso elimina una variable que sabes que es importante | Variables **sin escalar**: la que tiene números pequeños necesita un coeficiente grande y paga más multa | Poner `StandardScaler` **antes** del modelo (dentro de un pipeline) |
| Subí `C` "para regularizar más" y el modelo sobreajusta más | Confundir `C` con `alpha` | Recordar: `C` grande = **menos** regularización |
| Todos los coeficientes son casi cero y el modelo predice siempre lo mismo | `alpha` demasiado grande → subajuste | Bajar `alpha`; buscarlo con validación cruzada |
| Lasso elige variables distintas cada vez que cambian un poco los datos | Variables muy correlacionadas | Elastic Net o Ridge |
| Probé `alpha` = 1, 2, 3, 4, 5 y "no cambia nada" | Buscar en escala lineal | Buscar en **escala logarítmica**: 0.001, 0.01, 0.1, 1, 10, 100 |

El primero es tan común que merece su propio experimento.

### 🔮 Ejercicio 8 — Kilómetros contra metros
Dos variables **igual de importantes**: `distancia_m` (en metros, números grandes) y `tiempo_h` (en horas, números pequeños). Entrenamos Lasso **sin escalar** y **escalando**.

¿Cuál variable crees que Lasso eliminará si no escalamos?

✍️ *Mi predicción:*

In [ ]:
rng_e = np.random.default_rng(2)
n = 300
datos = pd.DataFrame({"distancia_m": rng_e.normal(5_000, 1_000, n),   # miles de metros
                      "tiempo_h":    rng_e.normal(2, 0.2, n)})        # unas pocas horas
# Ambas variables aportan lo mismo a y (una desviación típica de cada una suma 10 puntos)
y_e = 10 * (datos.distancia_m - 5_000) / 1_000 + 10 * (datos.tiempo_h - 2) / 0.2 + rng_e.normal(0, 3, n)

sin_escalar = Lasso(alpha=3).fit(datos, y_e)
escalando   = make_pipeline(StandardScaler(), Lasso(alpha=3)).fit(datos, y_e)
print("Sin escalar:", {c: round(float(v), 3) for c, v in zip(datos.columns, sin_escalar.coef_)})
print("Escalando  :", {c: round(float(v), 2) for c, v in zip(datos.columns, escalando[-1].coef_)})

<details><summary><b>👀 Ver respuesta</b></summary>

Sin escalar, Lasso **elimina `tiempo_h`** (coeficiente 0), aunque es tan importante como la distancia. ¿Por qué? Como `tiempo_h` se mueve en décimas de hora, necesita un coeficiente **grande** (≈ 50) para influir, y la multa de L1 es proporcional al tamaño del coeficiente. A `distancia_m` le basta un coeficiente diminuto (0.01), así que casi no paga multa.

Escalando, las dos quedan en igualdad de condiciones y reciben coeficientes **casi iguales** (≈ 7.0 y 6.8).

**Regla:** siempre que regularices un modelo lineal (o uses KNN, SVM o redes), **escala primero**. Y hazlo dentro de un pipeline para no tener fugas (Lección 7.5).
</details>

---
## 8. 📋 Ficha del modelo

| | Ridge (L2) | Lasso (L1) | Elastic Net |
|---|---|---|---|
| **Tipo** | Regresión (y logística con `C`) | Regresión | Regresión |
| **Intuición** | Cobra por kilo al cuadrado: encoge todo | Cobra por kilo: apaga lo que no aporta | Un poco de cada |
| **Hiperparámetros clave** | `alpha` | `alpha` | `alpha`, `l1_ratio` |
| **✅ Fortalezas** | Estable; maneja bien variables correlacionadas; rápido | Selección automática de variables; modelos pequeños e interpretables | Selección de variables + estabilidad con correlacionadas |
| **❌ Debilidades** | No elimina variables | Inestable con variables correlacionadas | Una perilla más que ajustar |
| **Úsalo cuando…** | Muchas variables útiles, quizá correlacionadas | Sospechas que muchas columnas sobran | Muchas columnas, algunas correlacionadas |
| **Requisito** | **Escalar** las variables | **Escalar** las variables | **Escalar** las variables |

> 💡 Atajo práctico: `RidgeCV` y `LassoCV` buscan el mejor `alpha` con validación cruzada automáticamente. Las usarás en los ejercicios finales.

## 9. 🧠 Resumen en 5 puntos
1. **Regularizar** es añadir a la pérdida un **cobro por complejidad**: el modelo solo usa coeficientes grandes si de verdad valen la pena.
2. **L2 (Ridge)** encoge todos los coeficientes; **L1 (Lasso)** lleva muchos a **cero** y selecciona variables; **Elastic Net** mezcla ambos.
3. `alpha` es la tarifa: muy baja → sobreajuste, muy alta → subajuste. Se elige con **validación cruzada** y en **escala logarítmica**.
4. **`C` es el inverso de `alpha`**: en logística y SVM, `C` grande = menos regularización.
5. Regularizar **exige escalar** las variables; si no, castigas a las que usan números pequeños.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** Tu regresión logística tiene 100 % en entrenamiento y 82 % en prueba. ¿Qué cambio tiene **más** sentido?
- a) Subir `C` de 1 a 100
- b) Bajar `C` de 1 a 0.01
- c) Quitar el `StandardScaler`
- d) Entrenar más iteraciones

**2.** Tienes 500 columnas y sospechas que solo unas 20 importan. ¿Qué modelo pruebas primero?
- a) `Ridge` · b) `Lasso` · c) `LinearRegression` · d) Da igual

**3.** Con `alpha` enorme, un modelo Ridge predice…
- a) Lo mismo que una regresión sin regularizar
- b) Casi siempre el promedio de `y`
- c) Valores aleatorios
- d) Un error

**4.** ¿Cuál de estas búsquedas de `alpha` es la más razonable?
- a) 1, 2, 3, 4, 5 · b) 0.001, 0.01, 0.1, 1, 10, 100 · c) 100, 200, 300 · d) Solo 1

**5.** Lasso se queda con `metros_cuadrados` y tira `habitaciones` (muy correlacionadas). Al reentrenar con otra muestra, hace lo contrario. ¿Qué pruebas?
- a) Subir `alpha` · b) Elastic Net · c) Quitar el escalado · d) `LinearRegression`

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Es sobreajuste; en logística, **bajar** `C` = más regularización. a) haría lo contrario.
2. **b)** Lasso hace selección de variables: llevará a cero la mayoría de las columnas inútiles.
3. **b)** Todos los coeficientes tienden a cero y solo queda el intercepto, que es el promedio de `y`: subajuste.
4. **b)** Escala logarítmica: cada valor es 10 veces el anterior. El efecto de `alpha` se nota en órdenes de magnitud, no en sumas de 1.
5. **b)** Elastic Net reparte el peso entre variables correlacionadas y es más estable.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Un compañero dice: *"La regularización empeora el error de entrenamiento, así que empeora el modelo"*. ¿Qué le respondes?

**B. Caso de análisis 💼** Un hospital quiere predecir el riesgo de reingreso con **300 variables** del expediente. Los médicos piden un modelo que **use pocas variables** para poder revisarlas a mano. Tienes dos candidatos con el mismo error en validación cruzada:
- **Opción A:** Ridge que usa las 300 variables.
- **Opción B:** Lasso que usa 25.

¿Cuál recomiendas? ¿Qué riesgo de Lasso le advertirías al hospital?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Que el objetivo no es acertar en entrenamiento, sino **con datos nuevos**. La regularización empeora un poco el entrenamiento a cambio de mejorar (a menudo mucho) la prueba: cambia varianza por un poco de sesgo (Lección 4.5). Lo viste en la Sección 2.

**B.** La **Opción B**: mismo error y 25 variables que un médico puede revisar. Riesgo que hay que advertir: si hay variables **correlacionadas** (p. ej. dos análisis de sangre que miden casi lo mismo), Lasso puede haberse quedado con una "al azar"; que una variable no aparezca **no significa** que no tenga relación con el reingreso. Para elegir de forma más estable, prueba Elastic Net, y revisa si las 25 elegidas cambian mucho al reentrenar con otra muestra.
</details>

### 10.3 🧩 Práctica integradora: "¿Cuáles de estas 60 columnas sirven?"

Una clínica te manda los datos de diabetes que ya conoces, pero su sistema añadió **50 columnas basura** (números aleatorios sin relación con nada). No te dicen cuáles son. Tu misión: un modelo que **prediga bien** y que **descubra** qué columnas sirven.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Prepara los datos**

In [ ]:
rng_d = np.random.default_rng(42)
basura = pd.DataFrame(rng_d.normal(size=(len(X_diab), 50)),
                      columns=[f"col_basura_{i}" for i in range(50)])
X60 = pd.concat([X_diab, basura], axis=1)
X60_tr, X60_te, y60_tr, y60_te = train_test_split(X60, y_diab, test_size=0.25, random_state=0)
print(X60.shape, "→ entrenamiento:", X60_tr.shape[0], "· prueba:", X60_te.shape[0])

**Paso 2 🔮 — Sin regularizar.** Entrenamos una `LinearRegression` con las 60 columnas. ¿Cómo esperas que sea el R² (Lección 4.1) en entrenamiento comparado con el de prueba?

✍️ *Mi apuesta:*

In [ ]:
lineal = make_pipeline(StandardScaler(), LinearRegression()).fit(X60_tr, y60_tr)
print(f"R² entrenamiento: {lineal.score(X60_tr, y60_tr):.3f}")
print(f"R² prueba       : {lineal.score(X60_te, y60_te):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Entrenamiento ≈ **0.64**, prueba ≈ **0.19**: una brecha enorme. Las 50 columnas basura le dieron al modelo "libertad" para acomodar ruido. **Sobreajuste.**
</details>

**Paso 3 ▶️ — Ridge y Lasso con `alpha` elegido por validación cruzada.** `RidgeCV` y `LassoCV` prueban muchos `alpha` y se quedan con el mejor (usando solo el entrenamiento).

In [ ]:
ridge = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 50))).fit(X60_tr, y60_tr)
lasso = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0)).fit(X60_tr, y60_tr)

for nombre, m in [("Ridge", ridge), ("Lasso", lasso)]:
    print(f"{nombre}: alpha elegido = {m[-1].alpha_:8.3f}   "
          f"R² train = {m.score(X60_tr, y60_tr):.3f}   R² prueba = {m.score(X60_te, y60_te):.3f}")

✋ ¿Cuál gana en prueba? ¿Cuánto mejoró respecto a la regresión sin regularizar?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Los dos mejoran a la regresión sin regularizar (0.19): Ridge llega a ≈ **0.26** y Lasso a ≈ **0.32**. Lasso gana porque el problema es justo su especialidad: **la mayoría de las columnas sobran**. Además, la brecha entre entrenamiento y prueba se achica, sobre todo en Lasso.

> 📝 Los R² de prueba son modestos: predecir la evolución de la diabetes con 10 mediciones es difícil, y este conjunto de prueba (111 pacientes) resultó especialmente exigente. Lo que importa aquí es la **comparación** entre modelos.
</details>

**Paso 4 ✋ — ¿Qué columnas sobrevivieron?** Ejecuta y cuenta cuántas columnas reales y cuántas basura conservó Lasso.

In [ ]:
coefs = pd.Series(lasso[-1].coef_, index=X60.columns)
vivas = coefs[coefs != 0]
print(f"Lasso conservó {len(vivas)} de {len(coefs)} columnas")
print(f"  reales: {sum(~vivas.index.str.startswith('col_basura'))} de 10")
print(f"  basura: {sum(vivas.index.str.startswith('col_basura'))} de 50")
vivas.sort_values(key=abs, ascending=False).round(1).head(12)

<details><summary><b>👀 Ver respuesta</b></summary>

Lasso conservó **15** columnas: **6 de las 10 reales** y **9 de las 50 basura**. Eliminó más del 80 % de la basura sin que nadie le dijera cuál era. Las cuatro de mayor peso son reales (`bmi`, `s5`, `bp`, `s3`); las basura que sobrevivieron tienen coeficientes **pequeños** (como mucho ≈ 5).

No es perfecto: con solo 331 pacientes, algunas columnas basura "parecen" útiles por casualidad. Por eso una lista de variables seleccionadas es una **pista**, no una prueba.
</details>

**Paso 5 ✏️ — Tú decides la tarifa.** Cambia `ALPHA` y observa el compromiso entre **cuántas columnas** conserva Lasso y **cuánto acierta**. ¿Encuentras un valor que tire casi toda la basura sin perder mucho R²?

In [ ]:
ALPHA = 5      # ✏️ prueba 1, 3, 5, 8, 15

m = make_pipeline(StandardScaler(), Lasso(alpha=ALPHA)).fit(X60_tr, y60_tr)
coefs = pd.Series(m[-1].coef_, index=X60.columns)
vivas = coefs[coefs != 0].index
print(f"alpha={ALPHA}: R² prueba = {m.score(X60_te, y60_te):.3f} · "
      f"reales = {sum(~vivas.str.startswith('col_basura'))} · basura = {sum(vivas.str.startswith('col_basura'))}")

<details><summary><b>👀 Ver respuesta</b></summary>

| `ALPHA` | R² prueba | reales | basura |
|---|---|---|---|
| 1 | 0.259 | 9 | 39 |
| 3 | 0.317 | 6 | 17 |
| 5 | 0.325 | 5 | 6 |
| 8 | 0.331 | 4 | 1 |
| 15 | 0.309 | 4 | 0 |

Con `ALPHA = 8` Lasso tira casi toda la basura (queda 1) y logra el mejor R². Con 15 ya no queda basura, pero el modelo está tan restringido que el R² vuelve a bajar: subajuste. Fíjate en que también se fueron algunas columnas **reales** poco útiles: Lasso conserva lo que **más** aporta, no todo lo que aporta algo.

⚠️ **Ojo:** aquí elegiste mirando el conjunto de **prueba** para aprender. En un proyecto real, ese valor se elige con validación cruzada (como hizo `LassoCV`) y la prueba se usa una sola vez al final. Cómo buscar hiperparámetros sin hacer trampa es justo el tema de las lecciones 7.3 y 7.5.
</details>

---
## 📚 Para profundizar
- 📺 **Video:** StatQuest, "Regularization Part 1: Ridge Regression" y "Part 2: Lasso Regression" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Linear Models* (Ridge, Lasso, Elastic Net).
- 📖 **Libro:** *An Introduction to Statistical Learning* (James et al.), capítulo 6: el dibujo de "rombo vs. círculo" que explica por qué L1 produce ceros.

---
⬅️ **Anterior:** [6.7 Proyecto del módulo: El tasador automático 🏠](../m06-ensambles/6.7-proyecto-del-modulo.md) · ➡️ **Siguiente:** [7.2 Early stopping: saber cuándo parar](7.2-early-stopping.ipynb)